In [ ]:
import random
import pandas as pd

def generate_bit_cot_puzzle():
    """Generates an 8-bit compound bitwise puzzle with a full reasoning trace."""
    
    # 1. Define the secret rule: Circular Left Shift + XOR Mask
    shift_amt = random.randint(1, 4)
    xor_mask = random.randint(1, 255)
    
    def apply_rule(x_int):
        # Circular left shift for 8-bit number
        shifted = ((x_int << shift_amt) | (x_int >> (8 - shift_amt))) & 255
        # XOR with mask
        return shifted ^ xor_mask

    # 2. Generate examples
    examples = []
    for _ in range(4):
        val = random.randint(0, 255)
        examples.append((val, apply_rule(val)))
        
    # Target to solve
    target_val = random.randint(0, 255)
    target_out_int = apply_rule(target_val)
    
    # Format as 8-bit binary strings
    target_in_str = f"{target_val:08b}"
    target_out_str = f"{target_out_int:08b}"
    
    # 3. Construct the User Prompt
    prompt = "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers.\n"
    prompt += "The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT.\n"
    prompt += "Here are some examples of input -> output:\n"
    for inp, outp in examples:
        prompt += f"{inp:08b} -> {outp:08b}\n"
    prompt += f"\nNow, determine the output for: {target_in_str}"
    
    # 4. Construct the Chain-of-Thought (CoT) Trace
    cot = "<think>\n"
    cot += "To find the secret 8-bit transformation rule, I will test common bitwise operations: shifts, rotations, and XOR masks.\n"
    
    val_in, val_out = examples[0]
    cot += f"Let's analyze the first example: {val_in:08b} -> {val_out:08b}.\n"
    cot += f"1. Let's test for a circular left shift. If we shift {val_in:08b} left by {shift_amt}, we get "
    shifted_val = ((val_in << shift_amt) | (val_in >> (8 - shift_amt))) & 255
    cot += f"{shifted_val:08b}.\n"
    
    cot += f"2. This doesn't match the output {val_out:08b} yet. Let's see if an XOR mask is applied after the shift.\n"
    cot += f"   {shifted_val:08b} XOR {val_out:08b} = {xor_mask:08b}.\n"
    cot += f"   This reveals a potential XOR mask of {xor_mask:08b}.\n"
    
    cot += "3. Let's apply this compound rule (Circular Left Shift by " + str(shift_amt) + " THEN XOR with " + f"{xor_mask:08b}" + ") to the target: " + target_in_str + ".\n"
    
    target_shifted = ((target_val << shift_amt) | (target_val >> (8 - shift_amt))) & 255
    cot += f"   - Step A: Circular left shift {target_in_str} by {shift_amt} gives {target_shifted:08b}.\n"
    cot += f"   - Step B: {target_shifted:08b} XOR {xor_mask:08b} gives {target_out_str}.\n"
    cot += "</think>\n"
    
    # 5. Format exactly for the evaluation metric
    full_response = cot + f"\\boxed{{{target_out_str}}}"
    
    return {
        "prompt": prompt,
        "full_response": full_response
    }

# Generate 5,000 synthetic rows
print("Generating Bitwise synthetic dataset...")
synthetic_data = [generate_bit_cot_puzzle() for _ in range(5000)]
synthetic_df = pd.DataFrame(synthetic_data)
synthetic_df.to_csv("synthetic_bitwise_cot.csv", index=False)
print("Saved to synthetic_bitwise_cot.csv")